In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset
from fundus_vessels_toolkit.models.topology.losses import BranchDigraphMiner
from fundus_vessels_toolkit.models.topology.model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from train import DigraphGNNTrainer, DigraphGNNTrainerConfig

vscode_theme()


In [ ]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")
train_set, val_set, test_set = dataset.split_sets(train_ratio=0.7, val_ratio=0.15)

## Visualize result from pred table


In [ ]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


### Load model from checkpoint


In [ ]:
# checkpoint = torch.load("GNN-Topo-v1/c2kx8j5h/checkpoints/epoch=239-step=8880.ckpt")
# checkpoint = torch.load("GNN-Topo-v1/ft6svpfg/checkpoints/epoch=179-step=3420.ckpt")
checkpoint = torch.load("GNN-Topo-v1/96dxz1ex/checkpoints/epoch=99-step=1900.ckpt")

model = BranchDigraphModel(checkpoint["hyper_parameters"]["config"]["model"])
model.load_state_dict({k[6:]: v for k, v in checkpoint["state_dict"].items() if k.startswith("model.")})
model = model.cuda().eval()

In [ ]:
ID = 15
eval_set = test_set
sample_gt, gt_digraph = eval_set.get(ID, version="fvt", return_digraph=True)
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(sample_gt.cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

assert VBranchDigraph.has_fp_av_p(gt_digraph)
valid_branch = ~gt_digraph.branch_fp()
av_gt = gt_digraph.branch_av_class() <= 1
print(((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())
print(((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())


In [ ]:
from fundus_vessels_toolkit.utils.nnet.profiling import Profiler, profiler

Profiler.reset("torch_interp_bilinear")
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(sample_gt.cuda())
profiler("torch_interp_bilinear").print()


In [ ]:
m, pred_tree = eval_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
    show_gt_graph=False,
)
m

In [ ]:
STOP

In [ ]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_info(b1=86, sort_by_p=True).round(3).head(20)

In [ ]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []

eval_set = test_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        sample, gt_digraph = eval_set.get(i, version="fvt", return_digraph=True)
        assert VBranchDigraph.has_all_p(gt_digraph) and gt_digraph.graph is not None

        valid_branch = ~gt_digraph.branch_fp()
        av_gt = (gt_digraph.branch_av_class() <= 1)[valid_branch]
        od = Point.parse(sample.od_yx.tolist())

        art_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(gt_digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(gt_digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == gt_digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(sample.cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == gt_digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == gt_digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

Parent ACC


In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

In [ ]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

In [ ]:
np.argsort(pred_parent_acc)

Dir ACC


In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

In [ ]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

AV ACC


In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

In [ ]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

In [ ]:
np.argsort(pred_av_acc)